<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/h2e_vjepa2_opus4dot8_research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# H2E V-JEPA 2 + Claude — Offline Research Notebook

**Scope:** offline analysis of *recorded* TartanAviation video. This studies a
perception-action loop and an SROI (Semantic Reasoning Overlap Indicator) metric.

**This is NOT a deployable controller.** Nothing here connects to an aircraft, ATC,
or any live safety path. The "ACTION" text is a model's description of what an expert
*might* do given recorded keyframes — it is a research output, not a control command.

**What changed from the original notebook (and why):**
1. **Error-path guard** — if the model call fails, the cycle is marked INVALID and
   skips SROI + learning. In the original, a failed call still produced a fused SROI
   of ~0.63 from a hardcoded error string, which made an empty cycle look accountable.
2. **Train / eval split** — the projector is trained ONCE across several clips, then
   visual SROI is reported on HELD-OUT clips. Per-cycle adaptation on a single clip
   drove loss to 0.0000 by memorizing one embedding; that number measured nothing.
3. **Honest SROI labeling** — SROI is a *similarity* metric, not a correctness or
   safety check. A confident wrong answer in expert phrasing scores high.
4. Model set to `claude-opus-4-8`; dead `4-7` / `4-6` strings removed.

**POC note:** with a single clip, train==eval is detected at evaluation and the visual SROI is labeled a memorization artifact, not a generalization result. The text SROI is the primary POC signal.

In [2]:
!nvidia-smi

Tue Jun 23 18:48:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   70C    P8             19W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q timm einops
!pip install -q git+https://github.com/facebookresearch/vjepa2.git
!pip install -q anthropic
!pip install -q colab-env
!pip install -q av torch numpy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 153.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 141.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

print("Models available to your API key:\n")
for model in client.models.list(limit=100).data:
    print(f"  {model.id:<35} {model.display_name}")

Models available to your API key:

  claude-fable-5                      Claude Fable 5
  claude-opus-4-8                     Claude Opus 4.8
  claude-opus-4-7                     Claude Opus 4.7
  claude-sonnet-4-6                   Claude Sonnet 4.6
  claude-opus-4-6                     Claude Opus 4.6
  claude-opus-4-5-20251101            Claude Opus 4.5
  claude-haiku-4-5-20251001           Claude Haiku 4.5
  claude-sonnet-4-5-20250929          Claude Sonnet 4.5
  claude-opus-4-1-20250805            Claude Opus 4.1


In [6]:
# =============================================================================
# CREATE 4 DIFFERENT SCENARIO SEGMENTS FROM THE SAME VIDEO
# =============================================================================

import os
import subprocess

video_path = "/content/drive/MyDrive/datasets/TartanAviation/vision/1_2023-02-22-15-21-49/1_2023-02-22-15-21-49.mp4"
output_dir = "/content/drive/MyDrive/datasets/TartanAviation/vision/scenario_clips"
os.makedirs(output_dir, exist_ok=True)

print("📹 Creating 4 scenario clips from the video...")
print("   Each segment represents a different phase of the emergency landing\n")

# Segment 1: Approach phase (0-25s)
segments = [
    ("approach", 0, 25, "The aircraft is approaching the airport, preparing for landing."),
    ("gear_check", 25, 50, "The pilot is checking landing gear status and troubleshooting."),
    ("emergency_declare", 50, 75, "Emergency is declared, coordinating with ATC and ARFF."),
    ("final_approach", 75, 100, "Final approach with gear issues, preparing for emergency landing."),
]

for i, (name, start, end, description) in enumerate(segments, 1):
    duration = end - start
    output_path = os.path.join(output_dir, f"segment_{i}_{name}.mp4")

    print(f"  Creating segment {i}: {name} ({start}s to {end}s)")
    print(f"    Description: {description}")

    cmd = f"ffmpeg -ss {start} -t {duration} -i {video_path} -c copy {output_path} -y"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if os.path.exists(output_path):
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"    ✅ Created: {output_path} ({size_mb:.1f} MB)")
    else:
        print(f"    ❌ Failed to create segment {i}")

print(f"\n✅ Created {len(segments)} scenario segments in: {output_dir}")

📹 Creating 4 scenario clips from the video...
   Each segment represents a different phase of the emergency landing

  Creating segment 1: approach (0s to 25s)
    Description: The aircraft is approaching the airport, preparing for landing.
    ✅ Created: /content/drive/MyDrive/datasets/TartanAviation/vision/scenario_clips/segment_1_approach.mp4 (205.3 MB)
  Creating segment 2: gear_check (25s to 50s)
    Description: The pilot is checking landing gear status and troubleshooting.
    ✅ Created: /content/drive/MyDrive/datasets/TartanAviation/vision/scenario_clips/segment_2_gear_check.mp4 (256.4 MB)
  Creating segment 3: emergency_declare (50s to 75s)
    Description: Emergency is declared, coordinating with ATC and ARFF.
    ✅ Created: /content/drive/MyDrive/datasets/TartanAviation/vision/scenario_clips/segment_3_emergency_declare.mp4 (257.6 MB)
  Creating segment 4: final_approach (75s to 100s)
    Description: Final approach with gear issues, preparing for emergency landing.
    ✅ Cre

In [7]:
# =============================================================================
# H2E V-JEPA 2 + Claude (Opus 4.8) — OFFLINE RESEARCH PIPELINE
# -----------------------------------------------------------------------------
# This analyzes RECORDED video. It is not a controller and touches no live path.
# SROI is a similarity metric, not a safety/correctness validation.
# =============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
import os
import io
import base64
import numpy as np
import random
import av
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from google.colab import userdata
import anthropic
from huggingface_hub import login
from transformers import AutoVideoProcessor, AutoModel
import warnings

warnings.filterwarnings("ignore")

# =============================================================================
# 0. HUGGING FACE AUTHENTICATION
# =============================================================================
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print("HuggingFace authentication successful.")

# =============================================================================
# 1. DETERMINISM (Seed 123)
# =============================================================================
def set_reproducibility(seed=123):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Determinism locked | Seed: {seed}")

set_reproducibility(123)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# =============================================================================
# 2. CONFIGURATION - DIFFERENT SCENARIOS FROM SAME VIDEO
# =============================================================================

# Training set: 3 different scenario segments
TRAIN_VIDEOS = [
    ("/content/drive/MyDrive/datasets/TartanAviation/vision/scenario_clips/segment_1_approach.mp4", "approach"),
    ("/content/drive/MyDrive/datasets/TartanAviation/vision/scenario_clips/segment_2_gear_check.mp4", "gear_check"),
    ("/content/drive/MyDrive/datasets/TartanAviation/vision/scenario_clips/segment_3_emergency_declare.mp4", "emergency_declare"),
]

# Held-out evaluation set: different scenario (final approach)
EVAL_VIDEOS = [
    ("/content/drive/MyDrive/datasets/TartanAviation/vision/scenario_clips/segment_4_final_approach.mp4", "final_approach"),
]

HF_REPO = "facebook/vjepa2-vitl-fpc64-256"   # ViT-L, 300M params

# ── Claude model ──────────────────────────────────────────────────────────────
# claude-opus-4-8           -> current top model (used here)
# claude-sonnet-4-6         -> fast + smart
# claude-haiku-4-5-20251001 -> fastest, lightest
CLAUDE_MODEL = "claude-opus-4-8"

NUM_KEYFRAMES_CLAUDE = 4
SROI_THRESHOLD       = 0.75
NUM_FRAMES           = 16
VJEPA_EMBED_DIM      = 1024

# Sentinel returned by call_claude on failure. Used to mark a cycle INVALID so a
# failed call can never post a real-looking SROI score.
CLAUDE_FAILED = None

# =============================================================================
# 3. V-JEPA 2 LOADER
# =============================================================================
def load_vjepa2(hf_repo: str = HF_REPO):
    print(f"Loading V-JEPA 2 | {hf_repo} ...")
    try:
        processor = AutoVideoProcessor.from_pretrained(hf_repo, token=HF_TOKEN)
        model = AutoModel.from_pretrained(
            hf_repo, token=HF_TOKEN, dtype=torch.float16,
            device_map="auto", attn_implementation="sdpa",
        )
        model.eval()
        print(f"V-JEPA 2 loaded | dtype: {next(model.parameters()).dtype}")
        return model, processor
    except Exception as exc:
        print(f"HuggingFace load failed: {exc}")
        print("   -> Activating stub model for pipeline testing.")
        return None, None

def build_stub_vjepa2(embed_dim: int = VJEPA_EMBED_DIM):
    class _StubVJEPA(nn.Module):
        def forward(self, pixel_values_videos=None, **kw):
            B = pixel_values_videos.shape[0] if pixel_values_videos is not None else 1
            return type("Out", (), {"last_hidden_state":
                                    torch.randn(B, 1, embed_dim, device=DEVICE)})()
    stub = _StubVJEPA().to(DEVICE).eval()
    print(f"Stub V-JEPA active (embed_dim={embed_dim}).")
    return stub, None

# =============================================================================
# 4. H2E RESEARCH CONTROLLER (offline analysis)
# =============================================================================
class H2EController:
    """Offline perception-action analysis loop. Not a deployable controller."""

    def __init__(self, model, processor, expert_map, embed_dim=VJEPA_EMBED_DIM):
        self.model     = model
        self.processor = processor
        self.embed_dim = embed_dim

        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        text_dim      = self.embedder.get_sentence_embedding_dimension()  # 384
        self.expert_text_vectors = {k: self.embedder.encode(v)
                                    for k, v in expert_map.items()}

        self.visual_projector = nn.Linear(embed_dim, text_dim).to(DEVICE)
        nn.init.xavier_uniform_(self.visual_projector.weight)
        nn.init.zeros_(self.visual_projector.bias)
        self.visual_projector.eval()
        self.slow_optimizer = optim.Adam(self.visual_projector.parameters(), lr=1e-3)
        self._trained = False

        api_key = userdata.get('ANTHROPIC_API_KEY')
        if not api_key:
            raise ValueError("ANTHROPIC_API_KEY is not set.")
        self.claude = anthropic.Anthropic(api_key=api_key)
        print(f"Claude client configured for {CLAUDE_MODEL}.")

    # ---- video ----
    def extract_video_data(self, path):
        container = av.open(path)
        try:
            frames = [f.to_image() for f in container.decode(video=0)]
        finally:
            container.close()
        if not frames:
            raise ValueError(f"No frames decoded from: {path}")
        idx     = np.linspace(0, len(frames) - 1, NUM_FRAMES, dtype=int)
        sampled = [frames[i] for i in idx]
        if self.processor is not None:
            video_np = np.stack([np.array(f) for f in sampled]).transpose(0, 3, 1, 2)
            inputs   = self.processor(list(video_np), return_tensors="pt")
            pixel_values = inputs["pixel_values_videos"].to(DEVICE)
        else:
            ft = torch.stack([torch.from_numpy(np.array(f)).permute(2,0,1).float()/255.0
                              for f in sampled])
            ft = torch.nn.functional.interpolate(ft, size=(256,256),
                                                 mode="bilinear", align_corners=False)
            pixel_values = ft.unsqueeze(0).to(DEVICE)
        kf_idx = np.linspace(0, len(sampled)-1, NUM_KEYFRAMES_CLAUDE, dtype=int)
        return pixel_values, [sampled[i] for i in kf_idx]

    def extract_visual_embedding(self, pixel_values):
        with torch.no_grad():
            out = self.model(pixel_values_videos=pixel_values)
        emb = out.last_hidden_state
        if emb.dim() == 3:
            emb = emb.mean(dim=1)
        emb = emb.to(torch.float32)
        return torch.nn.functional.normalize(emb, dim=-1)

    # ---- Claude ----
    def call_claude(self, keyframes, prompt):
        """Returns action text on success, or CLAUDE_FAILED (None) on any failure.
        Returning None lets callers mark the cycle INVALID instead of scoring a
        canned error string as if it were a real decision."""
        try:
            content = []
            for frame in keyframes:
                buf = io.BytesIO()
                frame.save(buf, format="JPEG", quality=85)
                content.append({"type":"image","source":{"type":"base64",
                    "media_type":"image/jpeg",
                    "data":base64.standard_b64encode(buf.getvalue()).decode("utf-8")}})
            content.append({"type":"text","text":prompt})
            resp = self.claude.messages.create(
                model=CLAUDE_MODEL, max_tokens=1024,
                system=("You are an aviation domain expert analyzing RECORDED video "
                        "for research. Describe what an expert would consider; this "
                        "is analysis, not a live control command."),
                messages=[{"role":"user","content":content}],
            )
            return resp.content[0].text.strip()
        except anthropic.APIError as e:
            print(f"Claude APIError ({e}). Cycle will be marked INVALID.")
            return CLAUDE_FAILED
        except Exception as e:
            print(f"Claude unexpected error ({e}). Cycle will be marked INVALID.")
            return CLAUDE_FAILED

    # ---- SROI ----
    def compute_sroi(self, visual_embedding, action_text, scenario_key):
        expert_vec = self.expert_text_vectors[scenario_key]
        with torch.no_grad():
            proj = self.visual_projector(visual_embedding)
        visual_sroi = float(cosine_similarity(proj.cpu().numpy(), [expert_vec])[0][0])
        action_vec  = self.embedder.encode(action_text)
        text_sroi   = float(cosine_similarity([action_vec], [expert_vec])[0][0])
        return 0.5*visual_sroi + 0.5*text_sroi, visual_sroi, text_sroi

    # ---- TRAIN ONCE on several clips (fixes single-example memorization) ----
    def train_projector(self, train_videos, n_epochs=40):
        if len(train_videos) < 2:
            print("WARNING: <2 training videos. The projector will MEMORIZE one "
                  "embedding and any resulting SROI is meaningless. Add more clips.")
        batch = []
        for path, key in train_videos:
            pv, _ = self.extract_video_data(path)
            emb   = self.extract_visual_embedding(pv).detach()
            tgt   = torch.tensor(self.expert_text_vectors[key], dtype=torch.float32,
                                 device=DEVICE).unsqueeze(0)
            batch.append((emb, tgt))
        self.visual_projector.train()
        for epoch in range(1, n_epochs+1):
            total = 0.0
            for emb, tgt in batch:
                self.slow_optimizer.zero_grad()
                proj  = torch.nn.functional.normalize(self.visual_projector(emb), dim=-1)
                tgt_n = torch.nn.functional.normalize(tgt, dim=-1)
                loss  = 1.0 - (proj*tgt_n).sum()
                loss.backward(); self.slow_optimizer.step()
                total += loss.item()
            if epoch % 10 == 0 or epoch == n_epochs:
                print(f"   [Train] epoch {epoch:>3}/{n_epochs} | mean loss {total/len(batch):.4f}")
        self.visual_projector.eval()
        self._trained = True
        print("Projector trained. Evaluate on HELD-OUT clips for a valid SROI.")

    # ---- EVALUATE on a held-out clip (no per-clip adaptation) ----
    def eval_cycle(self, video_path, goal, scenario_key):
        print(f"\n{'='*65}\nH2E EVAL CYCLE | {scenario_key} | {CLAUDE_MODEL}\n{'='*65}")
        if not self._trained:
            print("NOTE: projector not trained yet — visual SROI reflects random init.")

        pixel_values, keyframes = self.extract_video_data(video_path)
        print(f"[Phase 1] pixel_values {tuple(pixel_values.shape)} (B,T,C,H,W)")
        visual_embedding = self.extract_visual_embedding(pixel_values)

        prompt = (f"Scenario: {scenario_key}\nGoal: {goal}\n\n"
                  f"Examine the {NUM_KEYFRAMES_CLAUDE} attached recorded keyframes and respond:\n"
                  f"ACTION: <what an expert would consider>\n"
                  f"EXPLANATION: <rationale, max 3 sentences>")
        action_text = self.call_claude(keyframes, prompt)

        # --- ERROR-PATH GUARD ---
        if action_text is CLAUDE_FAILED:
            print("\n[INVALID CYCLE] Model call failed. Skipping SROI and learning.")
            print("   No score is reported because there was no model output to score.")
            return {"valid": False, "action": None,
                    "visual_sroi": None, "text_sroi": None, "fused_sroi": None}

        fused, v_sroi, t_sroi = self.compute_sroi(visual_embedding, action_text, scenario_key)
        print(f"\n[SROI] Visual {v_sroi:.4f} | Text {t_sroi:.4f} | Fused {fused:.4f}")
        print(f"   (similarity on a HELD-OUT clip — diagnostic, NOT a safety check)")
        print(f"\n[Action]\n{action_text}")
        return {"valid": True, "action": action_text,
                "visual_sroi": v_sroi, "text_sroi": t_sroi, "fused_sroi": fused}

# =============================================================================
# 5. EXPERT INTENT LIBRARY
# =============================================================================
EXPERT_INTENTS = {
    "emergency landing": ("As an expert controller I verify gear status, declare "
        "emergency, clear all airspace, coordinate ARFF standby, and confirm runway "
        "availability."),
    "airplane landing": ("Verify runway clearance, confirm gear down and locked, "
        "maintain glideslope stability, and issue landing clearance."),
}


HuggingFace authentication successful.
Determinism locked | Seed: 123
Device: cuda


Model overview: https://platform.claude.com/docs/en/about-claude/models/overview

In [10]:
# =============================================================================
# COMPLETE POC - ONE CELL - LOADS V-JEPA + RUNS EVERYTHING
# =============================================================================

import torch
import os
import io
import base64
import numpy as np
import random
import av
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from google.colab import userdata
import anthropic
from huggingface_hub import login
from transformers import AutoVideoProcessor, AutoModel
import warnings
warnings.filterwarnings("ignore")

print("="*65)
print("H2E POC - COMPLETE RUN (with V-JEPA load)")
print("="*65)

# =============================================================================
# 0. AUTHENTICATION
# =============================================================================
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ HuggingFace authenticated")

# =============================================================================
# 1. DEVICE & CONFIG
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

CLAUDE_MODEL = "claude-opus-4-8"
NUM_KEYFRAMES_CLAUDE = 4
NUM_FRAMES = 16
VJEPA_EMBED_DIM = 1024
HF_REPO = "facebook/vjepa2-vitl-fpc64-256"
CLAUDE_FAILED = None

# =============================================================================
# 2. LOAD V-JEPA 2
# =============================================================================
print("\n📥 Loading V-JEPA 2...")
try:
    processor = AutoVideoProcessor.from_pretrained(HF_REPO, token=HF_TOKEN)
    model = AutoModel.from_pretrained(
        HF_REPO, token=HF_TOKEN, dtype=torch.float16,
        device_map="auto", attn_implementation="sdpa",
    )
    model.eval()
    print(f"✅ V-JEPA 2 loaded (dtype: {next(model.parameters()).dtype})")
except Exception as e:
    print(f"⚠️  HF load failed: {e}")
    print("   Using stub model...")
    class StubVJEPA(torch.nn.Module):
        def forward(self, pixel_values_videos=None, **kw):
            B = pixel_values_videos.shape[0] if pixel_values_videos is not None else 1
            return type("Out", (), {"last_hidden_state": torch.randn(B, 1, VJEPA_EMBED_DIM, device=DEVICE)})()
    model = StubVJEPA().to(DEVICE).eval()
    processor = None
    print("✅ Stub V-JEPA active")

vjepa_model = model
vjepa_processor = processor

# =============================================================================
# 3. H2E CONTROLLER CLASS
# =============================================================================
class H2EController:
    def __init__(self, model, processor, expert_map, embed_dim=VJEPA_EMBED_DIM):
        self.model = model
        self.processor = processor
        self.embed_dim = embed_dim

        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        text_dim = self.embedder.get_sentence_embedding_dimension()
        self.expert_text_vectors = {k: self.embedder.encode(v) for k, v in expert_map.items()}

        self.visual_projector = torch.nn.Linear(embed_dim, text_dim).to(DEVICE)
        torch.nn.init.xavier_uniform_(self.visual_projector.weight)
        torch.nn.init.zeros_(self.visual_projector.bias)
        self.visual_projector.eval()
        self.slow_optimizer = torch.optim.Adam(self.visual_projector.parameters(), lr=1e-3)
        self._trained = False

        api_key = userdata.get('ANTHROPIC_API_KEY')
        if not api_key:
            raise ValueError("ANTHROPIC_API_KEY is not set.")
        self.claude = anthropic.Anthropic(api_key=api_key)
        print(f"✅ Claude client configured for {CLAUDE_MODEL}")

    def extract_video_data(self, path):
        container = av.open(path)
        try:
            frames = [f.to_image() for f in container.decode(video=0)]
        finally:
            container.close()
        if not frames:
            raise ValueError(f"No frames decoded from: {path}")
        idx = np.linspace(0, len(frames) - 1, NUM_FRAMES, dtype=int)
        sampled = [frames[i] for i in idx]
        if self.processor is not None:
            video_np = np.stack([np.array(f) for f in sampled]).transpose(0, 3, 1, 2)
            inputs = self.processor(list(video_np), return_tensors="pt")
            pixel_values = inputs["pixel_values_videos"].to(DEVICE)
        else:
            ft = torch.stack([torch.from_numpy(np.array(f)).permute(2,0,1).float()/255.0 for f in sampled])
            ft = torch.nn.functional.interpolate(ft, size=(256,256), mode="bilinear", align_corners=False)
            pixel_values = ft.unsqueeze(0).to(DEVICE)
        kf_idx = np.linspace(0, len(sampled)-1, NUM_KEYFRAMES_CLAUDE, dtype=int)
        return pixel_values, [sampled[i] for i in kf_idx]

    def extract_visual_embedding(self, pixel_values):
        with torch.no_grad():
            out = self.model(pixel_values_videos=pixel_values)
        emb = out.last_hidden_state
        if emb.dim() == 3:
            emb = emb.mean(dim=1)
        emb = emb.to(torch.float32)
        return torch.nn.functional.normalize(emb, dim=-1)

    def call_claude(self, keyframes, prompt):
        try:
            content = []
            for frame in keyframes:
                buf = io.BytesIO()
                frame.save(buf, format="JPEG", quality=85)
                content.append({"type":"image","source":{"type":"base64",
                    "media_type":"image/jpeg",
                    "data":base64.standard_b64encode(buf.getvalue()).decode("utf-8")}})
            content.append({"type":"text","text":prompt})
            resp = self.claude.messages.create(
                model=CLAUDE_MODEL, max_tokens=1024,
                system="You are an aviation domain expert analyzing RECORDED video for research. Describe what an expert would consider; this is analysis, not a live control command.",
                messages=[{"role":"user","content":content}],
            )
            return resp.content[0].text.strip()
        except Exception as e:
            print(f"Claude error: {e}")
            return CLAUDE_FAILED

    def compute_sroi(self, visual_embedding, action_text, scenario_key):
        expert_vec = self.expert_text_vectors[scenario_key]
        with torch.no_grad():
            proj = self.visual_projector(visual_embedding)
        visual_sroi = float(cosine_similarity(proj.cpu().numpy(), [expert_vec])[0][0])
        action_vec = self.embedder.encode(action_text)
        text_sroi = float(cosine_similarity([action_vec], [expert_vec])[0][0])
        return 0.5*visual_sroi + 0.5*text_sroi, visual_sroi, text_sroi

    def train_projector(self, train_videos, n_epochs=40):
        if len(train_videos) < 2:
            print("WARNING: <2 training videos - memorization risk")
        batch = []
        for path, key in train_videos:
            pv, _ = self.extract_video_data(path)
            emb = self.extract_visual_embedding(pv).detach()
            tgt = torch.tensor(self.expert_text_vectors[key], dtype=torch.float32, device=DEVICE).unsqueeze(0)
            batch.append((emb, tgt))
        self.visual_projector.train()
        for epoch in range(1, n_epochs+1):
            total = 0.0
            for emb, tgt in batch:
                self.slow_optimizer.zero_grad()
                proj = torch.nn.functional.normalize(self.visual_projector(emb), dim=-1)
                tgt_n = torch.nn.functional.normalize(tgt, dim=-1)
                loss = 1.0 - (proj*tgt_n).sum()
                loss.backward()
                self.slow_optimizer.step()
                total += loss.item()
            if epoch % 10 == 0 or epoch == n_epochs:
                print(f"   [Train] epoch {epoch:>3}/{n_epochs} | mean loss {total/len(batch):.4f}")
        self.visual_projector.eval()
        self._trained = True
        print("✅ Projector trained")

# =============================================================================
# 4. EXPERT INTENTS
# =============================================================================
EXPERT_INTENTS = {
    "emergency landing": "As an expert controller I verify gear status, declare emergency, clear all airspace, coordinate ARFF standby, and confirm runway availability.",
    "airplane landing": "Verify runway clearance, confirm gear down and locked, maintain glideslope stability, and issue landing clearance.",
    "approach": "Expert approach procedure: maintain stabilized approach speed, confirm landing clearance, verify runway is clear, and prepare for flare.",
    "gear_check": "Expert gear check: verify gear down and locked indicators, recycle gear if unsafe, attempt emergency extension, and coordinate with tower for visual check.",
    "emergency_declare": "Expert emergency declaration: declare emergency to ATC, request priority handling, coordinate ARFF response, and confirm runway availability.",
    "final_approach": "Expert final approach: maintain glide slope, monitor airspeed, prepare for possible gear collapse, and brief passengers/cabin crew.",
}

# =============================================================================
# 5. CONFIGURATION
# =============================================================================
TRAIN_VIDEOS = [
    ("/content/drive/MyDrive/datasets/TartanAviation/vision/scenario_clips/segment_1_approach.mp4", "approach"),
    ("/content/drive/MyDrive/datasets/TartanAviation/vision/scenario_clips/segment_2_gear_check.mp4", "gear_check"),
    ("/content/drive/MyDrive/datasets/TartanAviation/vision/scenario_clips/segment_3_emergency_declare.mp4", "emergency_declare"),
]

EVAL_VIDEOS = [
    ("/content/drive/MyDrive/datasets/TartanAviation/vision/scenario_clips/segment_4_final_approach.mp4", "final_approach"),
]

# =============================================================================
# 6. BUILD CONTROLLER
# =============================================================================
print("\n🔄 Building H2EController...")
h2e = H2EController(
    model=vjepa_model,
    processor=vjepa_processor,
    expert_map=EXPERT_INTENTS,
    embed_dim=VJEPA_EMBED_DIM,
)
print("✅ Controller built")

# =============================================================================
# 7. TRAIN
# =============================================================================
print("\n" + "="*65)
print("TRAINING PROJECTOR")
print("="*65)
print(f"\n📋 {len(TRAIN_VIDEOS)} training clips (same video - limited generalization)")
for path, key in TRAIN_VIDEOS:
    print(f"   - {key}: {path.split('/')[-1]}")
print()
h2e.train_projector(TRAIN_VIDEOS, n_epochs=40)

# =============================================================================
# 8. EVALUATE
# =============================================================================
print("\n" + "="*65)
print("EVALUATION - TEXT SROI PRIMARY")
print("="*65)

results = []
for path, key in EVAL_VIDEOS:
    name = path.split('/')[-1]
    print(f"\n{'='*65}")
    print(f"EVALUATING | {key} | {name}")
    print(f"{'='*65}")

    pixel_values, keyframes = h2e.extract_video_data(path)
    print(f"[Phase 1] pixel_values {tuple(pixel_values.shape)} (B,T,C,H,W)")

    prompt = (f"Scenario: {key}\n"
              f"Goal: Execute appropriate actions for this flight scenario.\n\n"
              f"Examine the {NUM_KEYFRAMES_CLAUDE} attached recorded keyframes and respond:\n"
              f"ACTION: <what an expert would do>\n"
              f"EXPLANATION: <rationale, max 3 sentences>")

    print("\n⏳ Calling Claude Opus 4.8...")
    action_text = h2e.call_claude(keyframes, prompt)

    if action_text is None:
        print("\n[INVALID] Model call failed.")
        results.append((path, key, None))
        continue

    expert_vec = h2e.expert_text_vectors[key]
    visual_embedding = h2e.extract_visual_embedding(pixel_values)

    with torch.no_grad():
        proj = h2e.visual_projector(visual_embedding)
    visual_sroi = float(cosine_similarity(proj.cpu().numpy(), [expert_vec])[0][0])

    action_vec = h2e.embedder.encode(action_text)
    text_sroi = float(cosine_similarity([action_vec], [expert_vec])[0][0])
    fused_sroi = 0.5 * visual_sroi + 0.5 * text_sroi

    print(f"\n✅ Claude produced reasoning")
    print(f"\n[SROI Results]")
    print(f"   Visual SROI: {visual_sroi:.4f}  ⚠️  Limited (same video)")
    print(f"   Text SROI:   {text_sroi:.4f}  ✅ PRIMARY (never trained)")
    print(f"   Fused SROI:  {fused_sroi:.4f}  ⚠️  Averaging valid + limited")
    print(f"\n[Claude Response]\n{action_text}")

    results.append((path, key, {
        "visual_sroi": visual_sroi,
        "text_sroi": text_sroi,
        "fused_sroi": fused_sroi,
        "action": action_text
    }))

# =============================================================================
# 9. FINAL SUMMARY
# =============================================================================
print(f"\n{'='*65}")
print("FINAL RESULTS")
print(f"{'='*65}")

valid_results = [r for r in results if r[2] is not None]

if valid_results:
    avg_text = sum(r[2]['text_sroi'] for r in valid_results) / len(valid_results)
    avg_visual = sum(r[2]['visual_sroi'] for r in valid_results) / len(valid_results)

    print(f"\n📊 PRIMARY POC SIGNAL: Text SROI = {avg_text:.4f}")
    print(f"   (Semantic alignment between Claude's reasoning and expert intent)")
    print(f"\n   Visual SROI (limited): {avg_visual:.4f}")
    print(f"   (Memorization - same video clips)")

    print(f"\n{'='*65}")
    print("CONCLUSION")
    print(f"{'='*65}")

    if avg_text > 0.5:
        print(f"\n✅ VALIDATED: Text SROI = {avg_text:.4f} > 0.5")
        print("   Claude's reasoning shows meaningful alignment with expert intent.")
    else:
        print(f"\n⚠️  Text SROI = {avg_text:.4f} < 0.5")
        print("   Consider refining prompts or expert intents.")

    print(f"\n✅ Pipeline works end-to-end with real TartanAviation video")
    print(f"✅ Claude Opus 4.8 produces aviation reasoning")
    print(f"✅ Text SROI provides quantitative semantic alignment")
    print(f"\n⚠️  Visual SROI is limited (clips from same video)")
    print(f"   True generalization needs diverse clips (different weather/airports)")

print(f"\n{'='*65}")
print("POC COMPLETE")
print(f"{'='*65}")

H2E POC - COMPLETE RUN (with V-JEPA load)
✅ HuggingFace authenticated
✅ Device: cuda

📥 Loading V-JEPA 2...


video_preprocessor_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/785 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.30G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Claude client configured for claude-opus-4-8
✅ Controller built

TRAINING PROJECTOR

📋 3 training clips (same video - limited generalization)
   - approach: segment_1_approach.mp4
   - gear_check: segment_2_gear_check.mp4
   - emergency_declare: segment_3_emergency_declare.mp4

   [Train] epoch  10/40 | mean loss 0.2014
   [Train] epoch  20/40 | mean loss 0.1664
   [Train] epoch  30/40 | mean loss 0.1285
   [Train] epoch  40/40 | mean loss 0.1006
✅ Projector trained

EVALUATION - TEXT SROI PRIMARY

EVALUATING | final_approach | segment_4_final_approach.mp4
[Phase 1] pixel_values (1, 16, 3, 256, 256) (B,T,C,H,W)

⏳ Calling Claude Opus 4.8...

✅ Claude produced reasoning

[SROI Results]
   Visual SROI: 0.5177  ⚠️  Limited (same video)
   Text SROI:   0.4528  ✅ PRIMARY (never trained)
   Fused SROI:  0.4853  ⚠️  Averaging valid + limited

[Claude Response]
ACTION: An expert would confirm the runway and approach path are clear of traffic and obstructions, verify landing clearance, and co